# Vie-GameEmo — Stage 0: Annotation Pipeline (Simplified)

**Input cần có:**
```
data/
├── raw_videos/         ← tất cả clips 5s .mp4 (chung 1 folder)
└── labels/
    ├── train.json      ← [{id, video, choice, confidence}]
    ├── val.json
    └── test.json
```

**Pipeline:**
```
1. Gộp labels → annotations + splits.json
2. Tiền xử lý: tách audio + frames
3. ASR: transcribe → cập nhật transcript vào annotations
4. (Tùy chọn) Multi-agent annotation: AU + VL + Audio + Consolidator
```

**Output:**
```
data/
├── annotations/*.json  ← Annotation JSON (có transcript, reasoning, ...)
├── splits.json
└── processed/          ← audio, frames (auto-generated)
```


In [ ]:
# ============================================================
# CELL 1 — Môi trường
# ============================================================
import os, sys
WORKING = os.getcwd()

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
else:
    print('CPU only')


In [ ]:
# ============================================================
# CELL 2 — CẤU HÌNH
# ============================================================

# --- Dataset ---
DATASET_INPUT = '/kaggle/input/vie-gameemo-dataset'
DATASET_LOCAL = os.path.join(WORKING, 'data')
PROJECT_INPUT = '/kaggle/input/vie-gameemo-code'

# --- ASR ---
ASR_BACKEND = 'whisper'           # 'whisper' | 'phowhisper'
WHISPER_MODEL = 'openai/whisper-large-v3'

# --- Multi-agent annotation (tùy chọn) ---
RUN_MULTI_AGENT = False           # True nếu muốn chạy Qwen-VL + Audio + Consolidator
ANNOTATION_MODEL = 'Qwen/Qwen2.5-7B-Instruct'


In [ ]:
# ============================================================
# CELL 3 — Cài thư viện
# ============================================================
%pip install -q \
    "numpy<2" \
    transformers>=4.45.0 \
    peft>=0.12.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.33.0 \
    faster-whisper>=1.0.3 \
    fasttext-wheel \
    pydantic>=2.0 \
    librosa \
    opencv-python-headless \
    scikit-learn \
    tiktoken \
    sentencepiece


In [ ]:
# ============================================================
# CELL 4 — Setup project
# ============================================================
import shutil, subprocess

PROJECT_DIR = os.path.join(WORKING, 'vie-gameemo-skeleton')

if os.path.exists(PROJECT_INPUT):
    if not os.path.exists(PROJECT_DIR):
        shutil.copytree(PROJECT_INPUT, PROJECT_DIR)
    print(f'Project: Kaggle input → {PROJECT_DIR}')
elif not os.path.exists(PROJECT_DIR):
    GITHUB_URL = 'https://github.com/rhy221/vie-gameemo-skeleton.git'
    subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, PROJECT_DIR], check=True)

if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    subprocess.run(['git', 'pull'], cwd=PROJECT_DIR, check=True)

SRC_DIR = os.path.join(PROJECT_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

if os.path.exists(DATASET_INPUT):
    DATA_DIR = DATASET_INPUT
else:
    DATA_DIR = DATASET_LOCAL

SCRIPTS = os.path.join(PROJECT_DIR, 'scripts')
CONFIG  = os.path.join(PROJECT_DIR, 'config.yaml')
print(f'Data:    {DATA_DIR}')
print(f'Scripts: {SCRIPTS}')


## Bước 1 — Gộp labels + Import annotations


In [ ]:
# ============================================================
# CELL 5 — Gộp train.json + val.json + test.json → annotations
# ============================================================
# import_labels.py đọc data/labels/{split}.json, tạo:
#   - data/annotations/*.json  (1 file per clip)
#   - data/splits.json         ({clip_id: "train"|"val"|"test"})
!python {SCRIPTS}/import_labels.py --data-root {DATA_DIR}


In [ ]:
# ============================================================
# CELL 5b — Verify: đếm annotations + phân phối
# ============================================================
import json
from pathlib import Path
from collections import Counter

annot_dir = Path(DATA_DIR) / 'annotations'
splits_path = Path(DATA_DIR) / 'splits.json'

annot_files = sorted(annot_dir.glob('*.json'))
print(f'Annotations: {len(annot_files)} files')

with open(splits_path, encoding='utf-8') as f:
    splits = json.load(f)
print(f'Splits: {dict(Counter(splits.values()))}')

# Phân phối nhãn
labels = []
for p in annot_files:
    data = json.loads(p.read_text(encoding='utf-8'))
    labels.append(data.get('emotion_label', '?'))
print(f'Labels: {dict(sorted(Counter(labels).items()))}')


## Bước 2 — Tiền xử lý (tách audio + frames)


In [ ]:
# ============================================================
# CELL 6 — Tách audio (wav 16kHz) + frames (4fps) từ clips
# ============================================================
# Clips nằm chung 1 folder: data/raw_videos/*.mp4
# Output: data/processed/audios/*.wav + data/processed/frames/{clip_id}/*.jpg
!python {SCRIPTS}/stage0_preprocess.py \
    --config {CONFIG} \
    --videos-dir {DATA_DIR}/raw_videos \
    --skip-webcam-detect


## Bước 3 — ASR Transcription


In [ ]:
# ============================================================
# CELL 7 — Whisper ASR → transcript → cập nhật annotations
# ============================================================
# Chạy Whisper trên audio, ghi transcript + language metadata
# vào annotation JSON.
!python {SCRIPTS}/transcribe.py --config {CONFIG}


In [ ]:
# ============================================================
# CELL 7b — Verify: xem vài transcript mẫu
# ============================================================
import json
from pathlib import Path

annot_dir = Path(DATA_DIR) / 'annotations'
samples = sorted(annot_dir.glob('*.json'))[:5]

n_has_transcript = 0
for p in sorted(annot_dir.glob('*.json')):
    data = json.loads(p.read_text(encoding='utf-8'))
    if data.get('transcript', '').strip():
        n_has_transcript += 1

print(f'Clips với transcript: {n_has_transcript}/{len(list(annot_dir.glob("*.json")))}')
print()

for p in samples:
    data = json.loads(p.read_text(encoding='utf-8'))
    t = data.get('transcript', '')[:80]
    lang = data.get('source_language', 'vi')
    label = data.get('emotion_label', '?')
    print(f'{p.stem}: [{label}] [{lang}] "{t}..."')


## Bước 4 — (Tùy chọn) Multi-agent Annotation

Chạy Qwen-VL + Qwen-Audio + Consolidator để sinh:
- `visual_objective_desc`: mô tả cảnh game
- `audio_tone_desc`: phân tích giọng nói
- `reasoning`: giải thích tại sao nhãn đó

**Cần GPU mạnh** (T4 16GB đủ cho 7B 4bit). Bỏ qua nếu chỉ cần train classifier.


In [ ]:
# ============================================================
# CELL 8 — Multi-agent annotation (tùy chọn)
# ============================================================
if RUN_MULTI_AGENT:
    !python {SCRIPTS}/stage0_annotate.py --config {CONFIG}
else:
    print('RUN_MULTI_AGENT=False — bỏ qua')
    print('Annotations hiện tại chỉ có: label + transcript')
    print('Đủ để train classifier. Reasoning sẽ rỗng cho đến khi chạy multi-agent.')


## Bước 5 — Lưu output


In [ ]:
# ============================================================
# CELL 9 — Archive annotations + processed để dùng cho training
# ============================================================
import zipfile

archive = os.path.join(WORKING, 'vie_gameemo_stage0.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Annotations
    annot_dir = os.path.join(DATA_DIR, 'annotations')
    for f in sorted(Path(annot_dir).glob('*.json')):
        zf.write(str(f), f'annotations/{f.name}')

    # Splits
    splits_file = os.path.join(DATA_DIR, 'splits.json')
    if os.path.exists(splits_file):
        zf.write(splits_file, 'splits.json')

    # Labels (backup)
    labels_dir = os.path.join(DATA_DIR, 'labels')
    if os.path.exists(labels_dir):
        for f in Path(labels_dir).glob('*.json'):
            zf.write(str(f), f'labels/{f.name}')

size_mb = os.path.getsize(archive) / 1e6
print(f'✅ Archive: {archive} ({size_mb:.1f} MB)')
print('📥 Download từ File browser → chuột phải → Download')
